# Day 079 — Exercise 5: SimpleAgent

**What you'll build:** the capstone class that binds tools + `llm_fn` and keeps a run history.

**Why it matters:** the module functions do the work; the class is a convenient binding layer so you set up the model and tools once, then call `.run()` many times. `add_tool` extends the agent at runtime — the start of an open-ended assistant.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing
    a runaway loop (a model that never says 'finish').
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── a safe calculator tool (no eval) ─────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate a basic arithmetic expression without eval().

    Supports + - * / ** % and parentheses. Anything else (names, calls,
    attribute access) raises ValueError. This is the safe way to give an
    agent a calculator: never eval() untrusted model output.
    """
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# ── the tool registry ────────────────────────────────────────────────────────
# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.
DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "word_count": {
        "description": "Count the words in a piece of text.",
        "parameters": {"text": "string - the text to count words in"},
        "fn": lambda args: str(len(str(args["text"]).split())),
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as a text block for the prompt."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)

# ── parsing messy LLM output ──────────────────────────────────────────────────
def safe_parse_json(text):
    """Extract and parse the first JSON object from messy LLM output.

    LLMs wrap JSON in markdown fences or prose. Instead of fighting that,
    slice from the first '{' to the last '}' and parse that. Returns a dict,
    or None if no valid JSON object is present.
    """
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def parse_action(text):
    """Turn raw LLM output into an action dict. NEVER raises.

    Returns one of:
      {"type": "tool",   "tool": name, "args": {...}}
      {"type": "finish", "answer": str}
    If the text is not a valid tool call, it falls back to a finish action
    holding the raw text - so a badly-formatted model reply still terminates
    the loop instead of crashing it.
    """
    data = safe_parse_json(text)
    if not isinstance(data, dict):
        return {"type": "finish", "answer": text.strip()}
    tool = data.get("tool")
    if tool and tool != "finish":
        return {"type": "tool", "tool": tool, "args": data.get("args", {})}
    return {"type": "finish", "answer": data.get("answer", text.strip())}

# ── executing tools + calling the model ──────────────────────────────────────
def execute_tool(action, tools):
    """Run one tool action against the registry. Returns a result string.

    Never raises: an unknown tool or a tool error is returned as text so the
    agent can read it and recover on its next turn.
    """
    name = action.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](action.get("args", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model. Inject llm_fn(messages) -> str for testing.

    llm_fn=None uses Ollama (llama3.2). A mock llm_fn lets the whole agent
    run offline with no model - which is how the tests drive the loop.
    """
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the agent loop ────────────────────────────────────────────────────────────
def build_agent_prompt(task, tools, history):
    """Build the [system, user] messages for one step of the loop."""
    system = "\n".join([
        "You are a tool-using agent. Solve the task by choosing ONE action at "
        "a time, returned as a single JSON object.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "To use a tool, reply with exactly:",
        '{"tool": "<name>", "args": {...}}',
        "When you know the final answer, reply with exactly:",
        '{"tool": "finish", "answer": "<answer>"}',
        "",
        "Reply with only the JSON object, nothing else.",
    ])
    lines = ["Task: " + str(task)]
    for step in history:
        lines.append("You called: " + json.dumps(step["action"]))
        lines.append("Result: " + str(step["result"]))
    lines.append("What is your next action?")
    user = "\n".join(lines)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


def run_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the agent loop until it finishes or hits max_iterations.

    The loop is the whole idea of an agent: build a prompt from the task and
    what has happened so far, ask the model for one action, run it, repeat.
    max_iterations is the safeguard - without it a confused model could loop
    forever. Returns:
      {"answer": str, "steps": [...], "iterations": int, "stopped": bool}
    stopped is True if the loop ran out of iterations without finishing.
    """
    if tools is None:
        tools = DEFAULT_TOOLS
    history = []
    for i in range(max_iterations):
        messages = build_agent_prompt(task, tools, history)
        response = call_llm(messages, llm_fn=llm_fn)
        action = parse_action(response)
        if action["type"] == "finish":
            return {"answer": action["answer"], "steps": history,
                    "iterations": i + 1, "stopped": False}
        result = execute_tool(action, tools)
        history.append({"action": action, "result": result})
    return {"answer": "Stopped: reached max_iterations without finishing.",
            "steps": history, "iterations": max_iterations, "stopped": True}


## Task

`SimpleAgent(tools=None, llm_fn=None, max_iterations=10)`

1. `__init__` — store `dict(DEFAULT_TOOLS if tools is None else tools)` (**copy**, so `add_tool` doesn't mutate the global), plus `_llm_fn`, `max_iterations`, and `_history = []`.
2. `add_tool(name, description, fn, parameters=None)` — add `{'description','parameters','fn'}` to `self.tools`; return `self`.
3. `run(task)` — call `run_agent` with the bound tools/llm_fn; append `{'task','result'}` to `_history`; return the result.
4. `history()` — return `list(self._history)` (a copy).
5. `clear_history()` — `self._history.clear()` (in place).

## Your Implementation

In [ ]:
class SimpleAgent:
    """A minimal tool-using agent binding tools + llm_fn."""

    def __init__(self, tools=None, llm_fn=None, max_iterations=10):
        raise NotImplementedError

    def add_tool(self, name, description, fn, parameters=None):
        raise NotImplementedError

    def run(self, task):
        raise NotImplementedError

    def history(self):
        raise NotImplementedError

    def clear_history(self):
        raise NotImplementedError


In [ ]:

# ── the agent as a class ──────────────────────────────────────────────────────
class SimpleAgent:
    """A minimal tool-using agent.

    Binds a tool registry and an optional llm_fn at construction, then runs
    tasks through run_agent and keeps a history of every run.

    Example::

        agent = SimpleAgent(llm_fn=my_llm_fn)
        result = agent.run("What is 2 + 2?")
        print(result["answer"])
    """

    def __init__(self, tools=None, llm_fn=None, max_iterations=10):
        # copy so add_tool never mutates the shared DEFAULT_TOOLS global
        self.tools = dict(DEFAULT_TOOLS if tools is None else tools)
        self._llm_fn = llm_fn
        self.max_iterations = max_iterations
        self._history = []

    def add_tool(self, name, description, fn, parameters=None):
        """Register a new tool. fn takes an args dict and returns a result."""
        self.tools[name] = {"description": description,
                            "parameters": parameters or {},
                            "fn": fn}
        return self

    def run(self, task):
        """Run one task through the agent loop. Returns the result dict."""
        result = run_agent(task, tools=self.tools, llm_fn=self._llm_fn,
                           max_iterations=self.max_iterations)
        self._history.append({"task": task, "result": result})
        return result

    def history(self):
        """Return a copy of the run history."""
        return list(self._history)

    def clear_history(self):
        """Clear the run history in place."""
        self._history.clear()


## Automated checks

In [ ]:

score, total = 0, 6
try:
    script = ['{"tool": "calculator", "args": {"expression": "2+2"}}',
              '{"tool": "finish", "answer": "4"}']
    agent = SimpleAgent(llm_fn=_make_mock_llm(script))
    out = agent.run('what is 2+2?')
    assert out['answer'] == '4'
    score += 1; print("✅ SimpleAgent.run returns the final answer")

    assert 'calculator' in agent.tools and 'word_count' in agent.tools
    score += 1; print("✅ SimpleAgent uses DEFAULT_TOOLS by default")

    agent.add_tool('shout', 'Uppercase text.',
                   lambda args: str(args['text']).upper(), {'text': 'string'})
    assert 'shout' in agent.tools
    assert 'shout' not in DEFAULT_TOOLS
    score += 1; print("✅ add_tool registers a tool without mutating DEFAULT_TOOLS")

    assert len(agent.history()) == 1
    score += 1; print("✅ history records each run")

    h = agent.history(); h.clear()
    assert len(agent.history()) == 1
    score += 1; print("✅ history() returns a copy, not the live list")

    agent.clear_history()
    assert len(agent.history()) == 0
    score += 1; print("✅ clear_history empties the log")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── the agent as a class ──────────────────────────────────────────────────────
class SimpleAgent:
    """A minimal tool-using agent.

    Binds a tool registry and an optional llm_fn at construction, then runs
    tasks through run_agent and keeps a history of every run.

    Example::

        agent = SimpleAgent(llm_fn=my_llm_fn)
        result = agent.run("What is 2 + 2?")
        print(result["answer"])
    """

    def __init__(self, tools=None, llm_fn=None, max_iterations=10):
        # copy so add_tool never mutates the shared DEFAULT_TOOLS global
        self.tools = dict(DEFAULT_TOOLS if tools is None else tools)
        self._llm_fn = llm_fn
        self.max_iterations = max_iterations
        self._history = []

    def add_tool(self, name, description, fn, parameters=None):
        """Register a new tool. fn takes an args dict and returns a result."""
        self.tools[name] = {"description": description,
                            "parameters": parameters or {},
                            "fn": fn}
        return self

    def run(self, task):
        """Run one task through the agent loop. Returns the result dict."""
        result = run_agent(task, tools=self.tools, llm_fn=self._llm_fn,
                           max_iterations=self.max_iterations)
        self._history.append({"task": task, "result": result})
        return result

    def history(self):
        """Return a copy of the run history."""
        return list(self._history)

    def clear_history(self):
        """Clear the run history in place."""
        self._history.clear()
```

**Why `dict(DEFAULT_TOOLS ...)`?** Without the copy, `self.tools` would *be* the module-level `DEFAULT_TOOLS`, so `add_tool` would leak into every other agent. Copying gives each agent its own registry.

</details>